# Territory reuse and spatial regularity

Does persistence of territorial locations contribute to increasing spatial regularity through selective retention, spatial anchoring, or both?

This notebook is being developed incrementally. Spatial computations belong in `compute_reuse_regularity.R`, which sources `spatial_analysis_functions.R` and will save results to `processed_data`. This Python notebook reads those results for statistics and plotting. The current notebook increment only inspects existing saved NND and previous-core tables; the R script currently contains setup and file discovery. Matching, local metrics, plots, regressions, and null models are not implemented yet.

Marking size is excluded as an explanatory variable. The drone measurements supply only the reuse threshold; drone coordinates are not appended to the satellite time series.

## 1. Data inspection and preprocessing

### Existing sources

| Information | Source |
| --- | --- |
| Satellite territory centres | `rawdata/TalChhapar/*_COORDINATES/*_TC.csv`; `rawdata/Velavadar/*_COORDINATES/*_LEK1.csv` and `*_LEK2.csv` |
| Lek polygons | `rawdata/TalChhapar/TalChhapar_Area.shp`; `rawdata/Velavadar/Velavadar_Lek1_Area.shp` and `Velavadar_Lek2_Area.shp` |
| Drone-derived reuse threshold | `model_territory_sizes.ipynb`, using `rawdata/territory_sizes_2023.txt` |
| Cross-observation distances and reuse plots | `compute_crossyear_nnd.R`, `plot_crossyear_nnd.ipynb` |
| NND statistics | `compute_nnd_metrics.R`, `model_nnd_metrics.ipynb` |
| PCFs and peak counts | `compute_pcf.R`, `plot_pcf.ipynb`, `model_pcf.ipynb` |
| Shared spatial functions | `spatial_analysis_functions.R` |

### Inspection findings

| Lek | Observations | First month | Last month |
| --- | ---: | --- | --- |
| Tal Chhapar | 10 | 2012-10 | 2023-12 |
| Velavadar Lek 1 | 13 | 2013-12 | 2024-05 |
| Velavadar Lek 2 | 11 | 2014-02 | 2024-05 |

The CSV columns `pos_x` and `pos_y` already use EPSG:32643 (UTM metres). Transforming the longitude/latitude columns reproduces them to within 0.00003 m. No missing or duplicate coordinate pairs were found within these files. Existing polygon files use WGS84 longitude/latitude and must be transformed to the same projected CRS.

Labels such as `21MAR` mean March 2021. Existing workflows assign the first day of the month for ordering; this is not a known survey day. Transitions must follow the next available observation within each lek, including observations in the same year. A missing file is not evidence of zero territories.

The supplied boundaries are fixed per lek; date-specific survey coverage has not been established. Velavadar Lek 2 has 16 points outside its polygon in November 2016 and one in March 2021. After clipping, counts are 63 and 152 respectively, matching the saved NND output; the corresponding median NNDs also agree. All other satellite points lie inside their supplied polygons.

The initial analysis retains the previous observation's KDE-core restriction within the existing lek polygons. Results will concern reuse and spatial refinement within the previously occupied core. Analysis of the wider lek is deferred. The saved pointwise table contains only current territories admitted by this restriction.

### First code increment

`compute_reuse_regularity.R` contains the spatial setup, satellite file discovery, and drone-derived reuse threshold. It follows the RStudio script convention of `compute_crossyear_nnd.R` and loads the existing spatial helper functions directly. The threshold is approximately 5.538 m; marking measurements are used only to retrieve it. Spatial matching, local regularity calculations, and spatial null simulations will be added there incrementally and exported to `processed_data`.

This notebook uses Python, following `model_nnd_metrics.ipynb` and `plot_crossyear_nnd.ipynb`. It currently reads their existing saved outputs. It does not run the R script or recompute spatial quantities.


In [ ]:
import pandas as pd

The existing NND output supplies the observation inventory and counts after the original spatial preprocessing.

In [ ]:
## Load NND metrics
df = pd.read_csv('processed_data/nnd_ALL.csv')
df[['lek_id', 'data_label', 'date', 'nnd_count']]

### Previous-observation KDE core

For each transition, the existing R workflow clips territories to the lek polygon, estimates a KDE from the previous observation using `get_kde_sigma()`, and retains current territories in `make_kde_grid()`'s core. The script specifies `core_thresh = 0.2` and `kde_dimyx = 512`; the core contains grid cells whose estimated intensity is at least 20% of the maximum. This is a relative-intensity contour, not a region containing 20% of the probability mass.

The loading code below is reused from `plot_crossyear_nnd.ipynb`. `data` contains transition-level counts, and `pointwise` contains the admitted current territories and their saved distances to previous territories. These distances are not one-to-one assignments. The saved tables do not include the KDE grid itself; `compute_reuse_regularity.R` will reconstruct that grid for spatial nulls by calling the existing `get_kde_sigma()`, `make_kde_grid()`, and `subset_points_to_kde_core()` functions.

`core_fraction` is the fraction of current, polygon-clipped territories inside the previous core. It describes the coverage of our restricted analysis, not the fraction reused. These are counts of territory observations across transitions, not unique territories or males.

In [ ]:
## Saved transitions and current territories within the previous KDE core
data = pd.read_csv('processed_data/crossyear_nnd_summary.csv', parse_dates=['date_now', 'date_prev'])
pointwise = pd.read_csv('processed_data/crossyear_nnd_pointwise.csv', parse_dates=['date_curr', 'date_prev']).rename(columns={'date_curr': 'date_now'})

In [ ]:
data['core_fraction'] = data['n_curr_in_prev_core'] / data['n_curr']
data[['lek_id', 'date_prev', 'date_now', 'n_curr', 'n_curr_in_prev_core', 'core_fraction']]

## 2. Spatial matching across observations

The initial analysis uses the existing minimum-distance definition. For each current territory inside the previous KDE core, `compute_reuse_regularity.R` calls `nnd()` to find its distance to the nearest previous territory and classifies the location as reused when that distance is at most `reuse_thresh`. This describes observed location proximity, not individual male identity.

One-to-one matching is deferred for this exploratory analysis. Multiple current territories can share a nearest predecessor, so reused current locations need not equal the number of distinct previous locations retained. A location excluded by the core restriction must not automatically be labelled absent or non-retained. Testing selective retention will require a reuse outcome defined for previous territories.

The R script currently calculates these values inside the transition loop; exporting results for this notebook is pending.

## 3. Local regularity measurements and spatial maps

Pending implementation. Begin with absolute log deviation of each NND from its observation's median NND and the CV of the six nearest-neighbour distances. Inspect maps, distributions, boundary distances, and the number of eligible territories before modelling.

Six-neighbour distance CV measures radial spacing consistency; it does not measure the angular arrangement of neighbours. Low values alone do not establish hexagonal order. Whole-lek PCF structure remains a separate outcome. Boundary eligibility and the reference median for subset comparisons require explicit choices.

## 4. Selective retention analysis

Pending implementation. First compare local irregularity between retained and non-retained locations separately by lek and transition. Marking size is excluded as requested, so this analysis cannot establish whether an association is independent of marking size.

Any subsequent regression must address observation interval, boundary position, spatial intensity, and dependence among nearby territories. Associations alone will not establish causation.

## 5. Spatial anchoring analysis and null models

Pending implementation. Keep retained locations fixed and compare observed additions with equal-count placements within the previous observation's KDE core and the existing lek polygon. The same core restriction must apply to observed additions and simulated placements. Examine changes at the same retained focal territories, with adequate neighbours and comparable boundary eligibility.

The placement intensity, minimum spacing, and treatment of unmatched points near predecessors must be agreed before simulation. Candidate new placements must respect the reuse definition. Conditioning too closely on observed additions could remove the gap-filling signal being tested; an unrestricted uniform null could confound that signal with habitat use.

## 6. Summary plots comparing the two mechanisms

Pending implementation. Compare retention with equal-count subsets under non-spatial retention and addition with equal-count feasible placement nulls. Examine each lek separately, using Tal Chhapar as the stable reference, before considering pooled inference.

Relate these results to existing whole-lek NND and PCF summaries without treating reduced subset size or a change in the normalising median as structural improvement. No mechanism has been tested in the current increment.